# Print Data Shapes and Types

This notebook will iterate through all files in the `data` directory and print their shapes and data types. This is useful for verifying the structure of the data files.

In [1]:
import os
import numpy as np

# Define the data directory
data_dir = "data/Northeast/"

# Iterate through all files in the directory
for file_name in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file_name)
    
    # Check if the file is a .npy file
    if file_name.endswith('.npy'):
        try:
            data = np.load(file_path)
            print(f"File: {file_name}")
            print(f"Shape: {data.shape}")
            print(f"Data Type: {data.dtype}")
            print("-" * 40)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            print("-" * 40)
    else:
        print(f"Skipping non-npy file: {file_name}")
        print("-" * 40)

File: DEM_northeast.npy
Shape: (64, 80)
Data Type: float64
----------------------------------------
File: uv100_test.npy
Shape: (7324, 2, 64, 80)
Data Type: float64
----------------------------------------
Skipping non-npy file: normalization_params.npz
----------------------------------------
File: 1000zt_test.npy
Shape: (7324, 2, 64, 80)
Data Type: float64
----------------------------------------
File: 1000zt_train.npy
Shape: (43824, 2, 64, 80)
Data Type: float64
----------------------------------------
File: uv100_train.npy
Shape: (43824, 2, 64, 80)
Data Type: float64
----------------------------------------


In [10]:
import xarray as xr

ds = xr.open_dataset("data/uv100.grib", engine="cfgrib")
print(list(ds.data_vars))
print(ds.coords)                 # 看有哪些坐标
print(ds.dims)                   # 看有哪些维度
print(ds["time"].values[:-5])     # 看前5个时间（如果有 time）

temp = xr.open_dataset("data/temp.grib", engine="cfgrib")
print(temp["time"].values[:-5])

geo  = xr.open_dataset("data/geo.grib", engine="cfgrib")
print(geo["time"].values[:-5])


['u100', 'v100']
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 409kB 2020-01-01 ... 2025-11-01T06:00:00
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 520B 54.0 53.75 53.5 ... 38.5 38.25 38.0
  * longitude   (longitude) float64 648B 116.0 116.2 116.5 ... 135.5 135.8 136.0
    valid_time  (time) datetime64[ns] 409kB ...
FrozenMappingWarningOnValuesAccess({'time': 51151, 'latitude': 65, 'longitude': 81})
['2020-01-01T00:00:00.000000000' '2020-01-01T01:00:00.000000000'
 '2020-01-01T02:00:00.000000000' ... '2025-10-31T23:00:00.000000000'
 '2025-11-01T00:00:00.000000000' '2025-11-01T01:00:00.000000000']
['2020-01-01T00:00:00.000000000' '2020-01-01T01:00:00.000000000'
 '2020-01-01T02:00:00.000000000' ... '2025-11-13T19:00:00.000000000'
 '2025-11-13T20:00:00.000000000' '2025-11-13T21:00:00.000000000']
['2020-01-01T00:00:00.000000000' '2020-01-01T01:00:00.000000000'
 '2020-01-01T02:00:00.000000000' ... 

In [11]:
import numpy as np

a = np.load("data/Northeast/1000zt_test.npy", mmap_mode="r")  # mmap 不会一次性读进内存
print("shape:", a.shape)
print("dtype:", a.dtype)

# 第0个小时、两个通道的二维场
print("t0 geo map shape:", a[0, 0].shape)
print("t0 temp map shape:", a[0, 1].shape)

# 看一小块数值（左上角 3x5）
print("t0 geo patch:\n", a[0, 0, :3, :5])
print("t0 temp patch:\n", a[0, 1, :3, :5])


shape: (8760, 2, 64, 80)
dtype: float64
t0 geo map shape: (64, 80)
t0 temp map shape: (64, 80)
t0 geo patch:
 [[1.96389181 1.95659234 1.93978659 1.90566583 1.85321152]
 [1.96643814 1.95200896 1.928413   1.88359767 1.8158654 ]
 [1.96304304 1.93707051 1.90821215 1.8661129  1.80703814]]
t0 temp patch:
 [[-1.52868962 -1.55110482 -1.57570686 -1.61014972 -1.65046973]
 [-1.51310833 -1.53101315 -1.56449926 -1.60741616 -1.63953549]
 [-1.50080731 -1.5163886  -1.55985221 -1.60768951 -1.62914796]]


In [4]:
import xarray as xr

temp = xr.open_dataset("data/temp.grib", engine="cfgrib")
geo  = xr.open_dataset("data/geo.grib",  engine="cfgrib")

# 1) 先看各自有哪些变量（通常每个文件就1个主变量）
print("temp vars:", list(temp.data_vars))
print("geo  vars:", list(geo.data_vars))

tvar = list(temp.data_vars)[0]
gvar = list(geo.data_vars)[0]

# 2) 打印一些基础信息（像“头部”）
print("\n[temp]", temp[tvar])
print("\n[geo ]",  geo[gvar])

# 3) 取“前3个时间点 + 左上角 4x6 网格”输出
temp_small = temp[tvar].isel(time=slice(0, 3), latitude=slice(0, 4), longitude=slice(0, 6))
geo_small  = geo[gvar ].isel(time=slice(0, 3), latitude=slice(0, 4), longitude=slice(0, 6))

print("\n=== temp small (time 0..2, lat 0..3, lon 0..5) ===")
print(temp_small)

print("\n=== geo small (time 0..2, lat 0..3, lon 0..5) ===")
print(geo_small)

# 4) 如果你想看“某个具体时刻的一张图的数值矩阵”
print("\n=== temp at time[0] grid patch ===")
print(temp[tvar].isel(time=0, latitude=slice(0, 4), longitude=slice(0, 6)).values)

print("\n=== geo at time[0] grid patch ===")
print(geo[gvar].isel(time=0, latitude=slice(0, 4), longitude=slice(0, 6)).values)


AttributeError: 'str' object has no attribute 'removeprefix'

In [ ]:
import numpy as np

a = np.load("data/Northeast/1000zt_train.npy", mmap_mode="r")   # 或 npz 里取 data
# 如果是 npz:
# d = np.load("1000zt_test.npz"); a = d["data"]

for c, name in enumerate(["geo", "temp"]):
    x = a[:, c]
    print(name, "mean", float(np.mean(x)), "std", float(np.std(x)),
          "min", float(np.min(x)), "max", float(np.max(x)))


geo mean 0.0027457118794995397 std 1.0010890446730554 min -5.571792339652888 max 3.690361720505827
temp mean -0.002973422938432205 std 0.9924934900235932 min -3.521909869990521 max 2.548257337811505
